<a href="https://colab.research.google.com/github/thanhsonnguyen-gif/ai-text-detection/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Chuẩn bị dữ liệu · Chủ: Người 1 · Phục vụ Chương 4
Chạy **một lần**. Sau khi xong, KHÔNG chạy lại trừ khi cả hai đồng ý (chạy lại sẽ đổi tập train/val/test và mọi kết quả cũ hết so sánh được).

Đầu ra trên Drive: `data/parquet/clean_train`, `clean_val`, `clean_test`, `attacked/attack=...`
Bảng/hình cho báo cáo: **T4.1** (thống kê dữ liệu), **F4.1** (phân bố độ dài).

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Java 21 | PySpark 4.0.5
Drive: /content/drive/MyDrive/BigData_IE212_R11_VB2/AI-Text-Project


## 1. Dữ liệu sạch (~800MB, khoảng 15–25 phút)

In [ ]:
from aitext import data, evaluate
t0 = time.time()
data.prepare_clean_split(spark, data.download("train_none"))
print(f"Xong sau {time.time()-t0:.0f}s")

KeyboardInterrupt: 

## 2. Thống kê dữ liệu → T4.1, F4.1

In [ ]:
train, val, test = data.load_splits(spark)
rows = []
for name, d in [("train", train), ("val", val), ("test", test)]:
    c = {r["label"]: r["count"] for r in d.groupBy("label").count().collect()}
    rows.append({"tập": name, "người": c.get(0, 0), "AI": c.get(1, 0), "tổng": c.get(0, 0) + c.get(1, 0)})
t41 = pd.DataFrame(rows); display(t41)
by_model = train.groupBy("model").count().orderBy(F.desc("count")).toPandas(); display(by_model)
by_domain = train.groupBy("domain", "label").count().toPandas().pivot(index="domain", columns="label", values="count"); display(by_domain)
lens = (train.withColumn("n_words", F.size(F.split("generation", r"\s+")))
             .groupBy("label").agg(F.avg("n_words").alias("số từ TB"), F.expr("percentile(n_words, 0.5)").alias("trung vị")).toPandas())
display(lens)
evaluate.save_table(t41, "T4.1_splits"); evaluate.save_table(by_model, "T4.1_by_model")
evaluate.save_table(by_domain.reset_index(), "T4.1_by_domain"); evaluate.save_table(lens, "T4.1_lengths")

In [ ]:
h = train.select("label", F.size(F.split("generation", r"\s+")).alias("n")).sample(0.3, seed=config.SEED).toPandas()
fig, ax = plt.subplots(figsize=(7, 3.5))
for lab, name in [(0, "người"), (1, "AI")]:
    ax.hist(h[h.label == lab]["n"].clip(upper=1500), bins=60, alpha=.6, density=True, label=name)
ax.set_xlabel("số từ"); ax.set_title("Phân bố độ dài văn bản (train)"); ax.legend()
evaluate.save_figure(fig, "F4.1_lengths"); plt.show()

## 3. Dữ liệu bị tấn công thật (~11,8GB, có thể hơn 1 giờ: nên chạy qua đêm)
Tải về ổ tạm của Colab, chỉ giữ phần thuộc tập test, lấy mẫu 2% mỗi kiểu tấn công, ghi Parquet lên Drive.

In [ ]:
   from aitext import data, evaluate
   import subprocess
   t0 = time.time()
   data.prepare_attacked_sample(spark, data.download("train"))
   n_empty = subprocess.run(["bash", "-c", f'find "{config.ATTACKED_DIR}" -name "*.parquet" -size 0 | wc -l'],
                            capture_output=True, text=True).stdout.strip()
   print("Số file rỗng:", n_empty)
   display(data.load_attacked(spark).groupBy("attack", "label").count().orderBy("attack", "label").toPandas())
   print(f"Xong sau {time.time()-t0:.0f}s")

Số file rỗng: 0


,attack,label,count
0,alternative_spelling,0,1567
1,alternative_spelling,1,1744
2,article_deletion,0,1550
3,article_deletion,1,1705
4,homoglyph,0,1531
5,homoglyph,1,1754
6,insert_paragraphs,0,1547
7,insert_paragraphs,1,1797
8,number,0,1574
9,number,1,1836


Xong sau 241s


In [ ]:
import shutil; shutil.rmtree(config.RAW_DIR, ignore_errors=True)   # dọn ổ tạm